# Canvas Course Downloader

Save your Canvas course material to your computer — no Terminal needed.

**Two ways to run it:**
- Click **▶ Run** on each cell top to bottom, **or**
- Use **Run All** — it's safe. Only the choice you set in Step 4 actually
  downloads; the other path is automatically skipped.

**Before you start:** this notebook must be in the **same folder** as
`get_course.py`.


## Step 1 — Install (run once)

Installs the one library the downloader needs. Only needed the first time.


In [ ]:
%pip install requests

## Step 2 — Your Canvas address

Change this only if your school isn't USD.


In [ ]:
BASE_URL = "https://sandiego.instructure.com"

# Load the downloader (get_course.py must be in this same folder)
import get_course as canvas
print("Ready. Using:", BASE_URL)

## Step 3 — Enter your access token

Get one in Canvas: **Account → Settings → + New Access Token** (copy it, it's
shown once).

Running this cell pops up a box to paste your token. Entering it here (instead
of saving it in the file) keeps your token private.


In [ ]:
from getpass import getpass
TOKEN = getpass("Paste your Canvas token, then press Enter: ")
print("Got it — token length:", len(TOKEN))

## Step 4 — Your choices (this is the only cell you edit)

Set these four values, then run the cell. Everything below reads from here, so
you can safely **Run All** — only the path you pick will download.

- **`MODE`** — `"one"` for a single course, or `"all"` for every course.
- **`COURSE_ID`** — your course number (only used when `MODE = "one"`). It's the
  number in the course web address: `.../courses/`**`12345`**.
- **`EVERYTHING`** — `True` for the full archive (all files, grades, your
  submitted work), or `False` for a lighter grab (pages, discussions,
  assignments, quizzes, syllabus).
- **`ACTIVE_ONLY`** — `True` to skip old/finished courses (only used when
  `MODE = "all"`).


In [ ]:
MODE        = "one"     # "one"  or  "all"
COURSE_ID   = "12345"   # your course number (used when MODE = "one")
EVERYTHING  = True      # True = full archive, False = lighter grab
ACTIVE_ONLY = False     # True = skip old courses (used when MODE = "all")

print(f"MODE = {MODE!r},  EVERYTHING = {EVERYTHING}")

## Step 5 — Download

Just run every cell below. Based on your `MODE`, exactly one path runs and the
others print a short "skipped" note. Your files land in a **`canvas_export`**
folder next to this notebook.


**Downloads ONE course** (runs only when `MODE = "one"`):

In [ ]:
if MODE != "one":
    print("Skipped: MODE is not 'one'.")
elif COURSE_ID.strip() in ("", "12345"):
    print("Set COURSE_ID in Step 4 to your real course number, then run again.")
else:
    canvas.run(
        base_url=BASE_URL, token=TOKEN, course_ids=[COURSE_ID],
        out_dir="./canvas_export", make_pdf=False,
        include_all_pages=EVERYTHING, include_files=True,
        include_syllabus=True, include_files_area=EVERYTHING,
        include_grades=EVERYTHING, include_submissions=EVERYTHING,
    )

**Lists ALL your courses** (runs only when `MODE = "all"`):

In [ ]:
my_courses = []
if MODE != "all":
    print("Skipped: MODE is not 'all'.")
else:
    client = canvas.CanvasClient(BASE_URL, TOKEN)
    my_courses = client.get_all_courses(include_concluded=not ACTIVE_ONLY)
    print(f"Found {len(my_courses)} course(s):")
    for c in my_courses:
        print("  ", c["id"], "-", c["name"])

**Downloads ALL your courses** (runs only when `MODE = "all"`):

In [ ]:
if MODE != "all":
    print("Skipped: MODE is not 'all'.")
elif not my_courses:
    print("No courses found — check the list cell above.")
else:
    canvas.run(
        base_url=BASE_URL, token=TOKEN,
        course_ids=[str(c["id"]) for c in my_courses],
        out_dir="./canvas_export", make_pdf=False,
        include_all_pages=EVERYTHING, include_files=True,
        include_syllabus=True, include_files_area=EVERYTHING,
        include_grades=EVERYTHING, include_submissions=EVERYTHING,
    )

---
Everything is saved in **`canvas_export`**, organized by course and module.
Open any `.html` file in your web browser.
